In [1]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# import seaborn as sb
import numpy as np
import datetime
import netCDF4
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import ScalarFormatter
import cartopy.mpl.ticker as cticker
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde


%matplotlib qt

datasets_dir = Path('../../../data/datasets/dataset_hplc_multi/')
save_dir = Path('../../../reports/data')
pigments_short = ['chlide', 'chla', 'chlb', 'chlc1 + c2', 'fucox', "19'hxfcx", "19'btfcx", 'diadino', 'allox', 'diatox', 'zeaxan', 'betac', 'peri']
pigments_short = ['chlide', 'chla', 'chlb', 'chlc12', 'fuco', "hex", "but", 'diad',
       'allo', 'diato', 'zea', 'caro', 'peri']


wv_13 = ['400', '412', '442', '490', '510', '560', '620', '665', '673', '681', '708']

save_dir.mkdir(parents=True, exist_ok=True)

In [2]:
x = pd.read_csv(datasets_dir/'rrs_lat_lon_month_season_depth_loc.csv')
y = pd.read_csv(datasets_dir/'pigments.csv')
print(len(y))

185


In [3]:
x.describe()

,400,412,442,490,510,560,620,665,673,681,...,November,December,spring,winter,autumn,summer,depth,med,black sea,med and black sea
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,...,185.0,185.0,185.000000,185.000000,185.0,185.000000,185.000000,185.000000,185.000000,185.000000
mean,0.005018,0.005582,0.006143,0.006599,0.005601,0.004335,0.001318,0.000886,0.000865,0.000861,...,0.0,0.0,0.854054,0.027027,0.0,0.118919,-709.367568,0.567568,0.313514,0.881081
std,0.001976,0.002331,0.002502,0.002723,0.002831,0.003585,0.001802,0.001215,0.001147,0.001169,...,0.0,0.0,0.354010,0.162602,0.0,0.324571,918.745106,0.496758,0.465180,0.324571
min,0.001167,0.001271,0.001406,0.002326,0.001966,0.000753,0.000102,0.000017,0.000027,0.000023,...,0.0,0.0,0.000000,0.000000,0.0,0.000000,-2851.000000,0.000000,0.000000,0.000000
25%,0.003542,0.003895,0.004186,0.004580,0.003545,0.001922,0.000395,0.000240,0.000248,0.000225,...,0.0,0.0,1.000000,0.000000,0.0,0.000000,-1314.000000,0.000000,0.000000,1.000000
50%,0.004767,0.005162,0.005805,0.006149,0.004481,0.002744,0.000660,0.000480,0.000485,0.000472,...,0.0,0.0,1.000000,0.000000,0.0,0.000000,-111.000000,1.000000,0.000000,1.000000
75%,0.006116,0.006829,0.007947,0.008445,0.007025,0.005680,0.001469,0.000988,0.000992,0.000977,...,0.0,0.0,1.000000,0.000000,0.0,0.000000,-53.000000,1.000000,1.000000,1.000000
max,0.010889,0.012242,0.013466,0.017220,0.019002,0.022411,0.014176,0.010422,0.009867,0.009629,...,0.0,0.0,1.000000,1.000000,0.0,1.000000,-12.000000,1.000000,1.000000,1.000000


In [4]:
y.describe()

,chlide_a[mg*m^3],chla[mg*m^3],chlb[mg*m^3],chlc1+c2[mg*m^3],fucox[mg*m^3],19'hxfcx[mg*m^3],19'btfcx[mg*m^3],diadino[mg*m^3],allox[mg*m^3],diatox[mg*m^3],zeaxan[mg*m^3],beta_car[mg*m^3],peridinin[mg*m^3]
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000
mean,0.162731,0.810745,0.045339,0.151623,0.318770,0.112247,0.016838,0.186608,0.028624,0.031112,0.038423,0.038602,0.030536
std,0.710862,1.673711,0.097423,0.469517,1.132058,0.113805,0.016789,0.688786,0.081034,0.069676,0.041310,0.141231,0.056404
min,0.000000,0.029000,0.001000,0.002000,0.001000,0.003000,0.002000,0.001000,0.000000,0.000300,0.003000,0.001000,0.000000
25%,0.005500,0.141200,0.005200,0.016400,0.009300,0.026700,0.005800,0.018800,0.002000,0.005000,0.015600,0.006300,0.003100
50%,0.011700,0.311200,0.022200,0.035200,0.032100,0.076600,0.011000,0.054800,0.006900,0.012700,0.028800,0.012900,0.006700
75%,0.023300,0.794300,0.043900,0.083400,0.121100,0.142600,0.019400,0.127300,0.024000,0.030400,0.043600,0.030000,0.028100
max,5.758500,16.615100,0.980600,4.571500,9.380900,0.519300,0.096800,7.969400,0.873200,0.673800,0.286900,1.795900,0.326500


# Data Description

In [5]:
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())  # You can choose different projections like 'PlateCarree'

# Add coastlines and other features
ax.coastlines()
# ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND, zorder=-1, edgecolor='black')
1# ax.add_feature(cfeature.LAKES, edgecolor='black')
# ax.add_feature(cfeature.RIVERS)

# Set the extent for the Mediterranean Sea
ax.set_extent([-11., 36, 30, 54], crs=ccrs.PlateCarree())  # [west, east, south, north]

# Plot the data as a scatter plot
sc = ax.scatter(x['lon'], x['lat'], c='royalblue', s=20, transform=ccrs.PlateCarree())


# Add gridlines with labels
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.8,
    color='gray',
    alpha=0.4,
    linestyle='--'
)

# Only label left and bottom axes (standard cartographic practice)
gl.top_labels = False
gl.right_labels = False

# Optional: control tick spacing
gl.xlocator = cticker.LongitudeLocator(5)
gl.ylocator = cticker.LatitudeLocator(5)

# Optional: format labels nicely
gl.xlabel_style = {'size': 22}
gl.ylabel_style = {'size': 22}


ax.text(
    0.5, -0.08, 'Longitude [degrees]',
    transform=ax.transAxes,
    ha='center', va='top',
    fontsize=25
)

ax.text(
    -0.1, 0.45, 'Latitude [degrees]',
    transform=ax.transAxes,
    ha='right', va='center',
    rotation=90,
    fontsize=25
)


plt.tight_layout()
plt.savefig(save_dir / 'sample_location.jpg', bbox_inches='tight', dpi=300)
plt.show()


In [6]:
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())  # You can choose different projections like 'PlateCarree'

# Add coastlines and other features
ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND, zorder=-1, edgecolor='black')
ax.add_feature(cfeature.LAKES, edgecolor='black')
ax.add_feature(cfeature.RIVERS)

# Set the extent for the Barcelona coasts
# ax.set_extent([-11., 36, 30, 54], crs=ccrs.PlateCarree())  # [west, east, south, north]

# Plot the data as a scatter plot
lons = [2.19, 2.2, 2.24]
lats = [41.37, 41.38, 41.41]
sc = ax.scatter(lons, lats, c='royalblue', s=20, transform=ccrs.PlateCarree())


# Add gridlines with labels
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.8,
    color='gray',
    alpha=0.4,
    linestyle='--'
)

# Only label left and bottom axes (standard cartographic practice)
gl.top_labels = False
gl.right_labels = False

# Optional: control tick spacing
gl.xlocator = cticker.LongitudeLocator(5)
gl.ylocator = cticker.LatitudeLocator(5)

# Optional: format labels nicely
gl.xlabel_style = {'size': 22}
gl.ylabel_style = {'size': 22}


ax.text(
    0.5, -0.08, 'Longitude [degrees]',
    transform=ax.transAxes,
    ha='center', va='top',
    fontsize=25
)

ax.text(
    -0.1, 0.45, 'Latitude [degrees]',
    transform=ax.transAxes,
    ha='right', va='center',
    rotation=90,
    fontsize=25
)


plt.tight_layout()
plt.savefig(save_dir / '4points.jpg', bbox_inches='tight', dpi=300)
plt.show()

In [46]:

fig, ax = plt.subplots()

for row in x[wv_13].values:
    ax.plot(wv_13, row, c='navy')
formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-2, -2))  # force 10^-2 notation
ax.yaxis.set_major_formatter(formatter)
ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))

# Optional: move the 10^-2 label to the top-left of the axis
ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))

plt.tight_layout()
plt.savefig(save_dir / 'spectral_plot.jpg', dpi=300,)

plt.show()

INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should

In [47]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Use seaborn style for consistency with histograms
plt.style.use("seaborn-v0_8-whitegrid")

# Make sure wavelengths are numeric for plotting
wv_13 = ['400', '412', '442', '490', '510', '560', '620', '665', '673', '681', '708']
wv_13_int = [400, 412, 442, 490, 510, 560, 620, 665, 673, 681, 708]

fig, ax = plt.subplots(figsize=(8, 5))

# Convert to numpy for easier math
spectra = x[wv_13].values
spectra_indx = np.max(spectra, axis=1)  < 0.015
spectra = spectra[spectra_indx]

# Compute mean and standard deviation spectra
mean_spectrum = np.mean(spectra, axis=0)
std_spectrum = np.std(spectra, axis=0)

#  separate by waters:
class1 = spectra[(np.argmax(spectra, axis=1)<=2)]
class2 = spectra[(np.argmax(spectra, axis=1)>2) * (np.argmax(spectra, axis=1)<=4)]
class3 = spectra[(np.argmax(spectra, axis=1)>4)]

# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class1.T,
    color="blue", alpha=0.6, linewidth=0.8, zorder=1
)
# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class2.T, 
    color="green", alpha=0.6, linewidth=0.8, zorder=1
)
# (Optional) Plot individual spectra as faint background lines
ax.plot(wv_13_int, class3.T,
    color="orange", alpha=0.6, linewidth=0.8, zorder=1
)
# Plot mean spectrum (main curve)
# ax.plot(
#     wv_13, mean_spectrum,
#     color="steelblue", linewidth=2.5, label="Mean spectrum", zorder=2
# )

# Add ±1σ shading
# ax.fill_between(
#     wv_13,
#     mean_spectrum - std_spectrum,
#     mean_spectrum + std_spectrum,
#     color="steelblue", alpha=0.2, zorder=1.5,
#     label="±1σ"
# )

# Format Y-axis with scientific notation (10⁻² × ...)
# formatter = ScalarFormatter(useMathText=True)
# formatter.set_powerlimits((-2, -2))
# ax.yaxis.set_major_formatter(formatter)
# ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, -2))
ax.set_ylim(0, 0.015)
ax.set_xlim(400, 708)
# Axis labels and title
ax.set_xlabel("Wavelength [nm]", fontsize=22)
ax.set_ylabel(r"$R_{rs}$ $[sr^{-1}]$", fontsize=22)
# ax.set_title("Spectral Reflectance of All Samples", fontsize=14, weight="bold")

# Ticks and legend styling
ax.tick_params(labelsize=22)
# ax.legend(fontsize=10, loc="best", frameon=True, fancybox=True)
# Create proxy lines for legend

legend_lines = [
    Line2D([0], [0], color='blue', lw=5, label='Case-1'),
    Line2D([0], [0], color='green', lw=5, label='Case-2a'),
    Line2D([0], [0], color='orange', lw=5, label='Case-2b'),
]

# Final layout
plt.tight_layout()
ax.legend(handles=legend_lines, fontsize=22, loc='best',     frameon=True,facecolor="white",edgecolor="black")

plt.savefig(save_dir / "spectral_plot_all.jpg", dpi=300, bbox_inches="tight")
plt.show()



In [32]:
spectra.max(axis=0)

array([0.0108895 , 0.01224167, 0.013466  , 0.01288445, 0.01302953,
       0.01345791, 0.00912859, 0.00551705, 0.00527164, 0.00537767,
       0.00728072])

In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, lognorm

bins = 30

# Optional: use a nice style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming your DataFrame is called `df` with 13 pigment columns
# e.g., df = pd.read_csv("pigments.csv")

df = y
# Create a figure with subplots
fig, axes = plt.subplots(5, 3, figsize=(12, 16))  # 4x4 grid (1 empty)
axes = axes.flatten()

# Plot histograms
for i, col in enumerate(df.columns):
    pig_short = pigments_short[i]
    ax = axes[i]
    df_filtered = df[df[col] <= 1]
    hist = sns.histplot(
        data=df_filtered, 
        stat="percent",
        x=col, 
        kde=False,         # adds a smooth density curve
        bins=bins,          # adjust as needed
        color="#4C72B0",
        common_norm=False,
        edgecolor="white",
        linewidth=0.5,
        alpha=0.7,
        ax=ax
    )
    # ax.text(0.7, 0.9, pig_short, transform=ax.transAxes, fontsize=11, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=1))  # box styling)
    # sns.kdeplot(
    #     data=df_filtered, 
    #     x=col,
    #     color="firebrick",  #
    #     linewidth=1.5,
    #     ax=ax,
    #     # cut=0
    # )
    # Calculating stats for the label
    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    
    stats_text = (f"$\\bf{{{pig_short}}}$")  # Bold Title
                  # f"$\\mu={mu:.2f}$\n"
                  # f"$\\sigma={sigma:.2f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    
    ax.set_xlabel("")  # remove x-label for cleanliness
    ax.set_ylabel("")  # remove x-label for cleanliness
    # ax.set_ylabel("KDE", fontsize=9)
    # ax.set_xlabel("Concentration [mg*m^3]", fontsize=9)
    ax.tick_params(labelsize=15)
    # ax.set_xlim(left=0)
    # 1. Fix the number of ticks on the left axis
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# Hide any empty subplots if 13 < 16
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

fig.text(0.5, 0.005, 'Pigment Concentration [mg $\cdot$ m$^{-3}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
# Adjust rect to make room for global labels if needed
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "pigment_histograms.jpg", dpi=300, bbox_inches="tight")
plt.show()

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup & Style
plt.style.use("seaborn-v0_8-whitegrid")

# Assuming x is your main dataframe and wv_13 is a list of column names
# df = x[wv_13] 
# FOR DEMO: I will create a dummy df. Replace this line with: df = x[wv_13]
import numpy as np
df = x[wv_13]

# 2. Figure Setup
fig, axes = plt.subplots(4, 3, figsize=(16, 16)) 
axes = axes.flatten()

# 3. Plotting Loop
for i, col in enumerate(df.columns):
    ax = axes[i]
    
    # --- Data Filtering ---
    # Rrs values are usually small (< 0.1). 
    # If you have outliers/flags (like 999), filter them here.
    # Otherwise, just use df[col] directly.
    df_filtered = df[df[col] <= 1]  # Keeping your filter logic (adjust threshold if needed)

    # --- A. Histogram ---
    hist = sns.histplot(
        data=df_filtered, 
        stat="percent",
        x=col, 
        kde=False,
        bins=bins,          
        color="#4C72B0",    # Matches previous graph (Standard Seaborn blue)
        edgecolor="white",  # Matches previous graph
        linewidth=0.5,
        alpha=0.7,
        ax=ax
    )
    
    # --- B. KDE (Density Curve) ---
    # sns.kdeplot(
    #     data=df_filtered,
    #     x=col,
    #     color="firebrick",  # Matches previous graph
    #     linewidth=1.5,
    #     ax=ax,
    #     cut=0
    # )

    # --- KDE scaled to percent ---
    data = df_filtered[col].dropna().values
    
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 500)
    
    bin_width = (data.max() - data.min()) / bins
    ys = kde(xs) * bin_width * 100
    
    ax.plot(xs, ys, color="firebrick", linewidth=2.5)
    
    # Using 'col' as the title. If you have a 'wavelengths_short' list, use that instead.
    stats_text = (f"$\\bf{{{col}}}nm$") 
                  # f"$\\mu={mu:.4f}$\n"  # Rrs is small, so used 4 decimal places
                  # f"$\\sigma={sigma:.4f}$")
    
    ax.text(0.95, 0.95, stats_text, 
            transform=ax.transAxes, 
            fontsize=15, 
            verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9, edgecolor="#dddddd"))

    # --- D. Formatting ---
    ax.set_xlabel("") 
    ax.set_ylabel("") 

    # Clean up ticks
    ax.tick_params(axis='both', which='major', labelsize=9)
    # ax.set_xlim(left=0) 
    left_ticks = np.linspace(0, ax.get_ylim()[1], 5)   # 5 ticks: 0%, 25%, 50%, 75%, 100%
    left_ticks = np.round(left_ticks, 1)
    ax.set_yticks(left_ticks)
    # 2. Create the twin axis
    # ax2 = ax.twinx()
    ax.tick_params(labelsize=15)
    
    # 3. Match the right axis ticks to the left axis count
    # patches = hist.patches
    # bin_width = patches[0].get_width()
    # right_ticks = left_ticks * bin_width * 100
    # ax2.set_ylim(0, right_ticks[-1])
    # right_ticks = np.round(right_ticks, 1)
    # ax2.set_yticks(right_ticks)

# 4. Cleanup Empty Axes
for j in range(len(df.columns), len(axes)):
    fig.delaxes(axes[j])

# 5. Global Labels
# Note: Changed unit to steradian inverse (sr^-1) which is standard for Rrs. 
# Change back to mg m^-3 if this is actually concentration.
fig.text(0.5, 0.005, 'Remote Sensing Reflectance [sr$^{-1}$]', ha='center', fontsize=15)
fig.text(0.005, 0.5, 'Normalized Occurrences [%]', va='center', rotation='vertical', fontsize=15)
# fig.text(0.95, 0.5, 'Normalized Occurrences [%]', va='center', rotation=270, fontsize=12)

plt.tight_layout()
plt.subplots_adjust(bottom=0.06, left=0.06, right=0.91)

plt.savefig(save_dir / "rrs_histograms.jpg", dpi=300, bbox_inches="tight")
plt.show()

# Data exploration

In [35]:
import pandas as pd

# Data extracted from Table S2
data = {
    "Pigment": [
        "19'-Butanoyloxyfucoxanthin", "19'-Hexanoyloxyfucoxanthin", "alpha-Carotene", "Alloxanthin",
        "beta-Carotene", "Chlorophyll b", "Chlorophyll c1", "Chlorophyll c2", "Chl c2-MGDG [14/18]",
        "Chlorophyll c3", "Chl c2-MGDG [14/14]", "Chlorophyllide a", "Cis-fucoxanthin",
        "Cis 19-hexanoyloxyfucoxanthin", "Diadinoxanthin", "Divinyl chlorophyll a", "Fucoxanthin",
        "Monovinyl chlorophyll a", "Monovinyl chlorophyll c3", "Monovinyl chlorophyll a allomer 1",
        "Monovinyl chlorophyll a allomer 2", "Monovinyl chlorophyll a epimer", "Neoxanthin",
        "Peridinin", "Prasinoxanthin", "Uriolide", "Violaxanthin", "Zeaxanthin"
    ],
    "Maximum (ng/l)": [
        136.37, 467.57, 49.96, 400.95, 58.45, 518.25, 41.81, 433.61, 109.75, 167.71,
        70.54, 609.08, 66.33, 43.45, 205.52, 47.95, 961.42, 3178.87, 26.91, 92.61,
        37.83, 22.75, 54.09, 222.72, 62.15, 40.4, 59.15, 104.27
    ],
    "Minimum (ng/l)": [
        1.25, 9.51, 0.1, 0.45, 1.46, 1.42, 0.05, 0.75, 0.26, 1.9, 0.53, 0.12, 0.0,
        0.43, 2.47, 0.1, 3.7, 58.03, 0.15, 0.08, 0.05, 0.49, 0.12, 0.47, 0.0, 0.0,
        0.22, 0.39
    ],
    "Average (ng/l)": [
        23.35, 73.36, 3.97, 18.79, 12.85, 57.05, 6.02, 55.66, 9.6, 31.77, 9.17,
        22.43, 5.88, 4.93, 30.42, 6.56, 90.31, 460.45, 3.88, 11.94, 7.83, 4.19,
        4.13, 8.24, 7.3, 3.53, 4.28, 21.24
    ],
    "SD (ng/l)": [
        18.21, 55.95, 4.75, 34.92, 8.23, 60.31, 7.13, 58.84, 16.7, 29.05, 9.74,
        67.11, 10.6, 5.88, 28.74, 9.24, 116.11, 375.5, 3.89, 12.12, 7.84, 3.8,
        5.23, 18.72, 8.53, 4.6, 5.42, 21.2
    ]
}

# Create a DataFrame
pigment_df = pd.DataFrame(data)

# Save to Excel
output_path = "HPLC_Pigment_Concentrations.xlsx"
pigment_df.to_csv(output_path, index=False)

print(f"Excel file saved successfully: {output_path}")


Excel file saved successfully: HPLC_Pigment_Concentrations.xlsx


In [36]:
pigment_hist = pd.read_csv('HPLC_Pigment_Concentrations.xlsx').set_index("Pigment")

In [37]:
pigment_hist

,Maximum (ng/l),Minimum (ng/l),Average (ng/l),SD (ng/l)
Pigment,,,,
19'-Butanoyloxyfucoxanthin,136.37,1.25,23.35,18.21
19'-Hexanoyloxyfucoxanthin,467.57,9.51,73.36,55.95
alpha-Carotene,49.96,0.10,3.97,4.75
Alloxanthin,400.95,0.45,18.79,34.92
beta-Carotene,58.45,1.46,12.85,8.23
Chlorophyll b,518.25,1.42,57.05,60.31
Chlorophyll c1,41.81,0.05,6.02,7.13
Chlorophyll c2,433.61,0.75,55.66,58.84
Chl c2-MGDG [14/18],109.75,0.26,9.60,16.70


In [17]:
from src.models.my_models import Model 

ImportError: cannot import name 'Model' from 'src.models.my_models' (C:\Users\sheic\PycharmProjects\pigment-retrieval-from-olci\src\models\my_models\__init__.py)

In [18]:
x, y = pd.read_csv(Path('../../../data/datasets/dataset_hplc_multi/rrs_lat_lon_month_season_depth_loc.csv')), pd.read_csv(Path('../../../data/datasets/dataset_hplc_multi/log_pigments.csv'))

In [19]:
x.columns

Index(['400', '412', '442', '490', '510', '560', '620', '665', '673', '681',
       '708', '778', '865', 'lat', 'lon', 'January', 'February', 'March',
       'April', 'May', 'June', 'July', 'August', 'September', 'October',
       'November', 'December', 'spring', 'winter', 'autumn', 'summer', 'depth',
       'med', 'black sea', 'med and black sea'],
      dtype='object')

In [20]:
# Extract features of sample
y_variables = ['chlide_a[mg*m^3]', 'chla[mg*m^3]', 'chlb[mg*m^3]', 'chlc1+c2[mg*m^3]',
                    'fucox[mg*m^3]', "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]", "diadino[mg*m^3]", "allox[mg*m^3]",
                    "diatox[mg*m^3]", "zeaxan[mg*m^3]", "beta_car[mg*m^3]", "peridinin[mg*m^3]"]

pigment_trad = {"19'btfcx[mg*m^3]": "19'-Butanoyloxyfucoxanthin", 
                "19'hxfcx[mg*m^3]": "19'-Hexanoyloxyfucoxanthin", 
                "allox[mg*m^3]": "Alloxanthin",
                "beta_car[mg*m^3]": "beta-Carotene", 
                'chlb[mg*m^3]': "Chlorophyll b", 
                'chlide_a[mg*m^3]': "Chlorophyllide a", 
                "diadino[mg*m^3]": "Diadinoxanthin", 
                'fucox[mg*m^3]': "Fucoxanthin", 
                "peridinin[mg*m^3]": "Peridinin", 
                "zeaxan[mg*m^3]": "Zeaxanthin"}
x_ds, y_ds = x[['400', '412', '442', '490', '510', '560', '620', '665', '673', '681',
       '708', '778', '865']],  y[y_variables]

In [21]:
y.describe()

,chlide_a[mg*m^3],chla[mg*m^3],chlb[mg*m^3],chlc1+c2[mg*m^3],fucox[mg*m^3],19'hxfcx[mg*m^3],19'btfcx[mg*m^3],diadino[mg*m^3],allox[mg*m^3],diatox[mg*m^3],zeaxan[mg*m^3],beta_car[mg*m^3],peridinin[mg*m^3]
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000
mean,-4.173526,-1.054053,-3.969451,-3.207373,-3.236831,-2.675606,-4.445453,-2.850473,-4.904824,-4.381398,-3.581293,-4.250820,-4.684801
std,1.646187,1.216083,1.290252,1.413500,1.886575,1.026980,0.825075,1.292006,1.576174,1.268148,0.757925,1.200320,1.531828
min,-5.999497,-2.833954,-5.809143,-5.262950,-5.809143,-4.341269,-5.626821,-4.637693,-6.907755,-6.319969,-4.774773,-6.023988,-6.907755
25%,-5.203007,-1.957578,-5.259097,-4.110474,-4.677741,-3.623092,-5.149897,-3.973898,-6.214608,-5.298317,-4.160484,-5.067206,-5.776353
50%,-4.448166,-1.167319,-3.807663,-3.346709,-3.438899,-2.569158,-4.509860,-2.904065,-4.976234,-4.366153,-3.547380,-4.350528,-5.005648
75%,-3.759302,-0.230294,-3.125841,-2.484107,-2.111139,-1.947712,-3.942482,-2.061209,-3.729701,-3.493313,-3.132698,-3.506558,-3.571986
max,1.750677,2.810312,-0.019591,1.519841,2.238676,-0.655274,-2.335108,2.075609,-0.135591,-0.394822,-1.248622,0.585506,-1.119325


In [22]:
x_ds.describe()

,400,412,442,490,510,560,620,665,673,681,708,778,865
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000
mean,0.005018,0.005582,0.006143,0.006599,0.005601,0.004335,0.001318,0.000886,0.000865,0.000861,0.000584,0.000218,0.000163
std,0.001976,0.002331,0.002502,0.002723,0.002831,0.003585,0.001802,0.001215,0.001147,0.001169,0.000963,0.000280,0.000193
min,0.001167,0.001271,0.001406,0.002326,0.001966,0.000753,0.000102,0.000017,0.000027,0.000023,0.000000,0.000000,0.000000
25%,0.003542,0.003895,0.004186,0.004580,0.003545,0.001922,0.000395,0.000240,0.000248,0.000225,0.000103,0.000022,0.000007
50%,0.004767,0.005162,0.005805,0.006149,0.004481,0.002744,0.000660,0.000480,0.000485,0.000472,0.000304,0.000146,0.000118
75%,0.006116,0.006829,0.007947,0.008445,0.007025,0.005680,0.001469,0.000988,0.000992,0.000977,0.000636,0.000304,0.000229
max,0.010889,0.012242,0.013466,0.017220,0.019002,0.022411,0.014176,0.010422,0.009867,0.009629,0.007281,0.001990,0.001203


In [23]:
# corr = np.corrcoef(x_ds.values.T)
# fig = plt.figure(figsize = (5,5))

# sb.heatmap(corr, square = True)
# plt.show()
